In [ ]:
import numpy as np
import pandas as pd
import cartopy.crs as ccrs
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon, Point
from shapely.ops import transform
import cartopy.feature as cfeature
import matplotlib.patheffects as pe
import textwrap
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import xarray as xr
import glob

In [2]:
# read in basin definition file
polygons_dict = {}

# read in basin definition file
with open("tc_basins.dat", "r") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        parts = line.split(",")
        basin_name = parts[0].replace('"', '')
        n_vertices = int(parts[1])

        lon_vals = list(map(float, parts[2:2+n_vertices]))
        lat_vals = list(map(float, parts[2+n_vertices:2+2*n_vertices]))

        coords = list(zip(lon_vals, lat_vals))
        poly = Polygon(coords)

        if basin_name not in polygons_dict:
            polygons_dict[basin_name] = []
        polygons_dict[basin_name].append(poly)

# Convert to GeoDataFrame
basin_records = []

for name, poly_list in polygons_dict.items():
    if len(poly_list) == 1:
        geom = poly_list[0]
    else:
        geom = MultiPolygon(poly_list)

    basin_records.append({
        "basin name": name,
        "geometry": geom
    })

basins = gpd.GeoDataFrame(basin_records, crs="EPSG:4326")

# fix invalid polygons
basins["geometry"] = basins["geometry"].buffer(0)

# remove empy geometries
basins = basins[~basins.geometry.is_empty]

# convert basins' lon to -180-180
basins["geometry"] = basins["geometry"].apply(
    lambda geom: transform(
        lambda x, y: (((x + 180) % 360) - 180, y),
        geom
    )
)

# read in NAtl subbasin polygons
sub_polygons_dict = {}

with open("tc_subbasins_NAtl_v5.dat", "r") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        parts = line.split(",")
        sub_basin_name = parts[0].replace('"', '')
        n_vertices = int(parts[1])

        lon_vals = list(map(float, parts[2:2+n_vertices]))
        lon_vals = [(lon + 180) % 360 - 180 for lon in lon_vals]
        lat_vals = list(map(float, parts[2+n_vertices:2+2*n_vertices]))

        coords = list(zip(lon_vals, lat_vals))
        poly = Polygon(coords)

        if sub_basin_name not in sub_polygons_dict:
            sub_polygons_dict[sub_basin_name] = []
        sub_polygons_dict[sub_basin_name].append(poly)

# Convert to GeoDataFrame
sub_basin_records = []

for name, poly_list in sub_polygons_dict.items():
    if len(poly_list) == 1:
        geom = poly_list[0]
    else:
        geom = MultiPolygon(poly_list)

    sub_basin_records.append({
        "sub_basin_name": name,
        "geometry": geom
    })

sub_basins = gpd.GeoDataFrame(sub_basin_records, crs="EPSG:4326",geometry="geometry")

# fix invalid polygons
sub_basins["geometry"] = sub_basins["geometry"].buffer(0)

# remove empty geometries
sub_basins = sub_basins[~sub_basins.geometry.is_empty]

# longitude conversion
import shapely.ops
def shift_lon(geom):
    return shapely.ops.transform(
        lambda x, y: (((x + 180) % 360) - 180, y),
        geom
    )

# shift lon
sub_basins["geometry"] = sub_basins["geometry"].apply(shift_lon)

In [3]:
# load wind datasets from ERA5
chunks = {"time": 30, "lat": 90, "lon": 140}

u200 = xr.open_dataset(
    "datasets/u-wind/post_processing/post_landmask/uwnd200_1940-2025_landmasked.nc",
    chunks=chunks
)
u850 = xr.open_dataset(
    "datasets/u-wind/post_processing/post_landmask/uwnd850_1940-2025_landmasked.nc",
    chunks=chunks
)
v200 = xr.open_dataset(
    "datasets/v-wind/post_processing/post_landmask/vwnd200_1940-2025_landmasked.nc",
    chunks=chunks
)
v850 = xr.open_dataset(
    "datasets/v-wind/post_processing/post_landmask/vwnd850_1940-2025_landmasked.nc",
    chunks=chunks
)

In [4]:
# convert lon to -180-180
u200 = u200.assign_coords(lon=(((u200.lon + 180) % 360) - 180)).sortby("lon")
u850 = u850.assign_coords(lon=(((u850.lon + 180) % 360) - 180)).sortby("lon")
v200 = v200.assign_coords(lon=(((v200.lon + 180) % 360) - 180)).sortby("lon")
v850 = v850.assign_coords(lon=(((v850.lon + 180) % 360) - 180)).sortby("lon")

# add CRS
u200 = u200.rio.write_crs("EPSG:4326")
u850 = u850.rio.write_crs("EPSG:4326")
v200 = v200.rio.write_crs("EPSG:4326")
v850 = v850.rio.write_crs("EPSG:4326")

# add spatial dims
u200 = u200.rio.set_spatial_dims(x_dim="lon", y_dim="lat")
u850 = u850.rio.set_spatial_dims(x_dim="lon", y_dim="lat")
v200 = v200.rio.set_spatial_dims(x_dim="lon", y_dim="lat")
v850 = v850.rio.set_spatial_dims(x_dim="lon", y_dim="lat")

In [5]:
# filter to N Atl
region = basins[basins["basin name"] == "N Atlantic"]
u200 = u200.rio.clip(region.geometry, region.crs, drop=True)
u850 = u850.rio.clip(region.geometry, region.crs, drop=True)
v200 = v200.rio.clip(region.geometry, region.crs, drop=True)
v850 = v850.rio.clip(region.geometry, region.crs, drop=True)

# filter to hurricane season
u200 = (
    u200
    .where(u200.time.dt.month.isin([6, 7, 8, 9, 10]), drop=True)
    .rio.clip(region.geometry, region.crs, drop=True)
)
u850 = (
    u850
    .where(u850.time.dt.month.isin([6, 7, 8, 9, 10]), drop=True)
    .rio.clip(region.geometry, region.crs, drop=True)
)
v200 = (
    v200
    .where(v200.time.dt.month.isin([6, 7, 8, 9, 10]), drop=True)
    .rio.clip(region.geometry, region.crs, drop=True)
)
v850 = (
    v850
    .where(v850.time.dt.month.isin([6, 7, 8, 9, 10]), drop=True)
    .rio.clip(region.geometry, region.crs, drop=True)
)

In [6]:
# select wind variables and remove the single pressure-level dimension
u200_new = u200["u"].squeeze("pressure_level", drop=True)
u850_new = u850["u"].squeeze("pressure_level", drop=True)
v200_new = v200["v"].squeeze("pressure_level", drop=True)
v850_new = v850["v"].squeeze("pressure_level", drop=True)

# rechunk the inputs to limit memory use
chunks = {"time": 30, "lat": 90, "lon": 140}

u200_new = u200_new.chunk(chunks)
u850_new = u850_new.chunk(chunks)
v200_new = v200_new.chunk(chunks)
v850_new = v850_new.chunk(chunks)

print("Wind inputs rechunked successfully")
print("u200 chunks:", u200_new.chunks)

Wind inputs rechunked successfully
u200 chunks: ((30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30

In [7]:
from pathlib import Path
out_dir = Path(
    "datasets/u-wind/post_processing/post_landmask/shear_annual"
)
out_dir.mkdir(parents=True, exist_ok=True)

years = np.unique(u200_new.time.dt.year.values)

for year in years:
    print(f"Calculating and saving {year}...", flush=True)

    # Select one year from each wind component
    u200_y = u200_new.sel(time=u200_new.time.dt.year == year)
    u850_y = u850_new.sel(time=u850_new.time.dt.year == year)
    v200_y = v200_new.sel(time=v200_new.time.dt.year == year)
    v850_y = v850_new.sel(time=v850_new.time.dt.year == year)

    # Calculate that year's wind shear
    shear_y = np.sqrt(
        (u850_y - u200_y)**2 +
        (v850_y - v200_y)**2
    ).rename("wind_shear")

    # Write that year's result to a separate file
    out_file = out_dir / f"shear_850_200_{year}.nc"
    shear_y.to_dataset().to_netcdf(out_file)

    print(f"Saved {out_file}", flush=True)

Calculating and saving 1940...
Saved datasets/u-wind/post_processing/post_landmask/shear_annual/shear_850_200_1940.nc
Calculating and saving 1941...
Saved datasets/u-wind/post_processing/post_landmask/shear_annual/shear_850_200_1941.nc
Calculating and saving 1942...
Saved datasets/u-wind/post_processing/post_landmask/shear_annual/shear_850_200_1942.nc
Calculating and saving 1943...
Saved datasets/u-wind/post_processing/post_landmask/shear_annual/shear_850_200_1943.nc
Calculating and saving 1944...
Saved datasets/u-wind/post_processing/post_landmask/shear_annual/shear_850_200_1944.nc
Calculating and saving 1945...
Saved datasets/u-wind/post_processing/post_landmask/shear_annual/shear_850_200_1945.nc
Calculating and saving 1946...
Saved datasets/u-wind/post_processing/post_landmask/shear_annual/shear_850_200_1946.nc
Calculating and saving 1947...
Saved datasets/u-wind/post_processing/post_landmask/shear_annual/shear_850_200_1947.nc
Calculating and saving 1948...
Saved datasets/u-wind/pos